# Download project assets

Set `HF_TOKEN` in the external credentials file, then run this notebook through `./scripts/run_colab_notebook.sh` from the repository root. Its model and lens files are uploaded under the configured R2 artifact prefix.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check boto3 "huggingface-hub==1.24.0"

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

In [ ]:
import shutil
from pathlib import Path

from huggingface_hub import hf_hub_download, snapshot_download

token = R2_CREDENTIALS.get("HF_TOKEN")
if not token:
    raise ValueError("Set HF_TOKEN in the external credentials file for asset download")
root = OUTPUT_DIR

model_dir = root / "assets/models/qwen3.5-4b"
snapshot_download(
    repo_id="Qwen/Qwen3.5-4B",
    revision="851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    local_dir=model_dir,
    token=token,
)

lens = hf_hub_download(
    repo_id="neuronpedia/jacobian-lens",
    revision="16a01f309fcec900fdcec3f4cd5b64f3d00e4d5a",
    filename="qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    token=token,
)
lens_destination = root / "assets/lenses/qwen3.5-4b/Qwen3.5-4B_jacobian_lens_n1000.pt"
lens_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(lens, lens_destination)

print(f"Assets ready in {root}")